In [ ]:
from langchain_openai import AzureChatOpenAI

model = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    temperature=0.2,
    callbacks=[lf_cb],
)


In [ ]:

import os


# ---- Langfuse (self-hosted) ----
os.environ.setdefault("LANGFUSE_PUBLIC_KEY",  "")
os.environ.setdefault("LANGFUSE_SECRET_KEY",  "")
# If you run Langfuse locally via docker compose the UI/API is on :3000 by default:
os.environ.setdefault("LANGFUSE_HOST",        "http://localhost:3000")

# ---- OpenAI ----
# Works with the standard OpenAI SDK interface but routed through Langfuse integration
os.environ.setdefault("OPENAI_API_KEY",       "")  # or use environment secrets

for k in ["OPENAI_API_KEY","LANGFUSE_PUBLIC_KEY","LANGFUSE_SECRET_KEY","LANGFUSE_BASE_URL"]:
    print(k, "=", ("set" if os.getenv(k) else "MISSING"))


OPENAI_API_KEY = set
LANGFUSE_PUBLIC_KEY = set
LANGFUSE_SECRET_KEY = set
LANGFUSE_BASE_URL = MISSING


In [6]:
# --- helper: fill {{var}} or {var} placeholders manually ---
def substitute_placeholders(messages_or_text, vars_):
    if isinstance(messages_or_text, list):  # chat messages
        new_msgs = []
        for m in messages_or_text:
            c = m["content"]
            for k, v in vars_.items():
                c = c.replace(f"{{{{{k}}}}}", str(v)).replace(f"{{{k}}}", str(v))
            new_msgs.append({**m, "content": c})
        return new_msgs
    else:  # single text prompt
        c = messages_or_text
        for k, v in vars_.items():
            c = c.replace(f"{{{{{k}}}}}", str(v)).replace(f"{{{k}}}", str(v))
        return c


In [7]:
# -------------------------------
# 0) Imports & shared utilities
# -------------------------------
import uuid
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

DEBUG_PREVIEW = True  # set False to disable preview prints

def run_prompt(prompt_name: str, label: str, variables: dict, model: ChatOpenAI, lf_cb: CallbackHandler):
    lf_client = get_client()
    prompt = lf_client.get_prompt(prompt_name, label=label)

    # Compile with vars (Langfuse), then ensure placeholders are actually substituted
    compiled = prompt.compile(variables=variables)          # may return list (chat) or str (text)
    compiled = substitute_placeholders(compiled, variables) # guarantee {{var}}/{var} filled

    # Optional preview for quick verification (no secrets)
    if DEBUG_PREVIEW:
        if isinstance(compiled, list):
            preview = "\n".join(f"{m.get('role','?')}: {str(m.get('content',''))[:240]}" for m in compiled)
        else:
            preview = str(compiled)[:600]
        print(f"\n--- PREVIEW: {prompt_name}:{label} ---\n{preview}\n--- END PREVIEW ---")

    trace_id = f"{prompt_name}-{label}-{uuid.uuid4()}"
    config = {
        "callbacks": [lf_cb],
        "langfuse": {
            "trace_args": {
                "id": trace_id,
                "name": f"{prompt_name}:{label}",
                "user_id": "udara-local",
                "tags": [f"prompt:{prompt_name}", f"label:{label}"],
                "metadata": {"prompt_version": getattr(prompt, "version", None)},
            }
        },
    }

    resp = model.invoke(compiled, config=config)
    print(f"\n=== {prompt_name}:{label} ===")
    print(resp.content)
    lf_client.flush()



In [8]:
# ==========================================
# 1) Common setup (Langfuse + OpenAI client)
# ==========================================
lf_client = get_client()
lf_cb = CallbackHandler()  # Langfuse v3.8.1: no kwargs
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, callbacks=[lf_cb])
parser = StrOutputParser()  # (kept for symmetry; not explicitly used here)


# =======================
# 2) Example 1: Summarizer
# =======================
summarizer_vars = {
    "source_text": """
Acme shipped Order #48291 in two parts due to inventory constraints. 
Customer claims double charge on 10/03 and requests refund for one transaction. 
Support ticket escalated to billing; resolution ETA 48 hours. Next meeting Friday 2 PM.
""",
    "tone": "neutral",
    "length": "5–7 sentences",
    "bullet_points": "yes",
}
run_prompt("demo-summarizer", "production", summarizer_vars, model, lf_cb)


--- PREVIEW: demo-summarizer:production ---
system: You are a precise summarizer. Write in a neutral tone.
Target length: 5–7 sentences. 
If yes == "yes", return a concise bulleted list. 
Preserve key facts, dates, figures, and decisions. Avoid fluff.

user: Summarize this text:
---

Acme shipped Order #48291 in two parts due to inventory constraints. 
Customer claims double charge on 10/03 and requests refund for one transaction. 
Support ticket escalated to billing; resolution ETA 48 hours. N
--- END PREVIEW ---

=== demo-summarizer:production ===
- Acme shipped Order #48291 in two parts due to inventory constraints.
- Customer reported a double charge on 10/03 and requested a refund for one transaction.
- The support ticket has been escalated to billing.
- Estimated time for resolution is 48 hours.
- Next meeting scheduled for Friday at 2 PM.


In [ ]:
# ==================================
# 3) Example 2: Conversation chat bot
# ==================================
conv_vars = {
    "brand": "Acme Gadgets",
    "language": "English",
    "knowledge": """Shipping takes 3–5 business days. 
Returns accepted within 30 days in original packaging. 
Warranty: 1 year for manufacturing defects; batteries excluded.""",
    "history_block": """User: Do you ship internationally?
Assistant: We currently ship only within the EU and UK.""",
    "user_input": "My device won’t hold charge. Is the battery covered?",
}
run_prompt("demo-conversation-bot", "production", conv_vars, model, lf_cb)





--- PREVIEW: demo-conversation-bot:production ---
system: You are a helpful, concise support assistant for Acme Gadgets.
Answer ONLY using the "Knowledge" below. If the answer is not covered, say you don't know and offer to escalate.
Respond in English.

Knowledge:
Shipping takes 3–5 business days
assistant: Conversation history so far:
User: Do you ship internationally?
Assistant: We currently ship only within the EU and UK.

user: My device won’t hold charge. Is the battery covered?
--- END PREVIEW ---

=== demo-conversation-bot:production ===
Batteries are excluded from the warranty, so they are not covered. If you have a manufacturing defect with the device itself, it may be covered under the 1-year warranty.


In [ ]:
# ==========================================
# 4) Example 3: Ticket triage (JSON classifier)
# ==========================================
triage_vars = {
    "issue_text": "I was charged twice for the same order yesterday. Please reverse one of the payments.",
    "customer_tier": "Gold",
    "recent_orders": "Order#94821 on 2025-11-02; Visa **** 2219",
}
run_prompt("demo-ticket-triage", "production", triage_vars, model, lf_cb)


--- PREVIEW: demo-ticket-triage:production ---
You are a ticket triage assistant. Read the issue and return STRICT JSON with keys:
- category: one of ["billing","shipping","returns","technical","account","other"]
- priority: one of ["low","medium","high","urgent"]
- route_to: one of ["billing-desk","shipping-desk","tech-desk","returns-desk","account-desk","general-queue"]
- rationale: short reason (max 30 words)

Context:
- Customer tier: Gold
- Recent orders: Order#94821 on 2025-11-02; Visa **** 2219

Issue:
I was charged twice for the same order yesterday. Please reverse one of the payments.

Return ONLY JSON, no extra text.

--- END PREVIEW ---

=== demo-ticket-triage:production ===
```json
{
  "category": "billing",
  "priority": "high",
  "route_to": "billing-desk",
  "rationale": "Customer was charged twice for a single order."
}
```


In [ ]:
!pip install wandb weave

In [ ]:
import uuid
import time
import wandb  # 👈 add this
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import weave

In [ ]:
wandb_run = wandb.init(
    project="llm-prompt-experiments",  # your project name
    name="local-prompt-testing",       # or something dynamic
    config={
        "model": "gpt-4o-mini",
        "environment": "local-dev",
    },
)


In [ ]:
def run_prompt(
    prompt_name: str,
    label: str,
    variables: dict,
    model: ChatOpenAI,
    lf_cb: CallbackHandler,
):
    lf_client = get_client()
    prompt = lf_client.get_prompt(prompt_name, label=label)

    # Compile with vars (Langfuse), then ensure placeholders are actually substituted
    compiled = prompt.compile(variables=variables)          # may return list (chat) or str (text)
    compiled = substitute_placeholders(compiled, variables) # guarantee {{var}}/{var} filled

    # Optional preview for quick verification (no secrets)
    if DEBUG_PREVIEW:
        if isinstance(compiled, list):
            preview = "\n".join(
                f"{m.get('role','?')}: {str(m.get('content',''))[:240]}"
                for m in compiled
            )
        else:
            preview = str(compiled)[:600]
        print(f"\n--- PREVIEW: {prompt_name}:{label} ---\n{preview}\n--- END PREVIEW ---")

    trace_id = f"{prompt_name}-{label}-{uuid.uuid4()}"
    config = {
        "callbacks": [lf_cb],
        "langfuse": {
            "trace_args": {
                "id": trace_id,
                "name": f"{prompt_name}:{label}",
                "user_id": "udara-local",
                "tags": [f"prompt:{prompt_name}", f"label:{label}"],
                "metadata": {
                    "prompt_version": getattr(prompt, "version", None),
                },
            }
        },
    }

    # 👉 measure latency around the actual model call
    start_time = time.time()
    resp = model.invoke(compiled, config=config)
    end_time = time.time()
    latency_ms = (end_time - start_time) * 1000.0

    print(f"\n=== {prompt_name}:{label} ===")
    print(resp.content)

    # ---------------------------
    # W&B: log prompt experiment + latency
    # ---------------------------
    # Try to get token usage if available from LangChain/OpenAI
    token_usage = {}
    try:
        # Depending on langchain_openai version, adjust this if needed
        token_usage = resp.response_metadata.get("token_usage", {})
    except Exception:
        token_usage = {}

    wandb.log(
        {
            # latency observability
            "latency_ms": latency_ms,

            # prompt experiment info
            "prompt_name": prompt_name,
            "label": label,
            "prompt_version": getattr(prompt, "version", None),
            "model_name": getattr(model, "model_name", None) or getattr(model, "model", None),

            # tokens (if available)
            "prompt_tokens": token_usage.get("prompt_tokens"),
            "completion_tokens": token_usage.get("completion_tokens"),
            "total_tokens": token_usage.get("total_tokens"),

            # you can also log something derived from variables
            "tone": variables.get("tone"),
            "length": variables.get("length"),
        }
    )

    lf_client.flush()
    return resp


In [ ]:
import uuid
import time
import wandb
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import weave  # optional if using W&B Weave decorators

DEBUG_PREVIEW = True  # toggle preview output

# 👇 Init W&B project and run
wandb_run = wandb.init(
    project="llm-prompt-experiments",
    name="local-prompt-testing",
    config={
        "model": "gpt-4o-mini",
        "environment": "local-dev",
    },
)

def substitute_placeholders(compiled, variables):
    # Simple fallback template substitution
    if isinstance(compiled, str):
        for k, v in variables.items():
            compiled = compiled.replace("{{" + k + "}}", str(v)).replace("{" + k + "}", str(v))
    elif isinstance(compiled, list):
        for msg in compiled:
            if isinstance(msg.get("content"), str):
                for k, v in variables.items():
                    msg["content"] = msg["content"].replace("{{" + k + "}}", str(v)).replace("{" + k + "}", str(v))
    return compiled

def run_prompt(
    prompt_name: str,
    label: str,
    variables: dict,
    model: ChatOpenAI,
    lf_cb: CallbackHandler,
):
    lf_client = get_client()
    prompt = lf_client.get_prompt(prompt_name, label=label)

    compiled = prompt.compile(variables=variables)
    compiled = substitute_placeholders(compiled, variables)

    if DEBUG_PREVIEW:
        if isinstance(compiled, list):
            preview = "\n".join(
                f"{m.get('role','?')}: {str(m.get('content',''))[:240]}"
                for m in compiled
            )
        else:
            preview = str(compiled)[:600]
        print(f"\n--- PREVIEW: {prompt_name}:{label} ---\n{preview}\n--- END PREVIEW ---")

    trace_id = f"{prompt_name}-{label}-{uuid.uuid4()}"
    config = {
        "callbacks": [lf_cb],
        "langfuse": {
            "trace_args": {
                "id": trace_id,
                "name": f"{prompt_name}:{label}",
                "user_id": "udara-local",
                "tags": [f"prompt:{prompt_name}", f"label:{label}"],
                "metadata": {
                    "prompt_version": getattr(prompt, "version", None),
                },
            }
        },
    }

    # 👇 W&B trace context for trace tree logging (optional)
    with wandb.trace(name=f"{prompt_name}:{label}") as span:
        start_time = time.time()
        resp = model.invoke(compiled, config=config)
        end_time = time.time()
        latency_ms = (end_time - start_time) * 1000.0

        print(f"\n=== {prompt_name}:{label} ===")
        print(resp.content)

        # 👇 Extract token usage if available
        token_usage = {}
        try:
            token_usage = resp.response_metadata.get("token_usage", {})
        except Exception:
            pass

        # 👇 Log prompt experiment + latency
        wandb.log({
            "latency_ms": latency_ms,
            "prompt_name": prompt_name,
            "label": label,
            "prompt_version": getattr(prompt, "version", None),
            "model_name": getattr(model, "model_name", None),
            "prompt_tokens": token_usage.get("prompt_tokens"),
            "completion_tokens": token_usage.get("completion_tokens"),
            "total_tokens": token_usage.get("total_tokens"),
            "tone": variables.get("tone"),
            "length": variables.get("length"),
        })

    lf_client.flush()
    return resp

# 👇 Example prompt variables
triage_vars = {
    "issue_text": "I was charged twice for the same order yesterday. Please reverse one of the payments.",
    "customer_tier": "Gold",
    "recent_orders": "Order#94821 on 2025-11-02; Visa **** 2219",
}

# 👇 You need to define these before calling `run_prompt`
model = ChatOpenAI(model_name="gpt-4o", temperature=0.2)
lf_cb = CallbackHandler()

# 👇 Trigger prompt run
run_prompt("demo-ticket-triage", "production", triage_vars, model, lf_cb)


In [ ]:
https://chatgpt.com/s/dr_691c5accfd0c81919064e18fb830b055

In [ ]:
import uuid
import time
import wandb
import weave  # 👈 W&B Weave

from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI

DEBUG_PREVIEW = True

# 1️⃣ --- Init W&B (for metrics + linking traces to a run) ---
wandb_run = wandb.init(
    project="llm-prompt-experiments",      # you can keep or change this
    name="local-prompt-testing-weave",     # run name
    config={
        "model": "gpt-4o-mini",
        "environment": "local-dev",
    },
)

# 2️⃣ --- Init Weave project (for trace UI) ---
# Use either "project-name" or "entity/project-name"
weave.init("llm-observability-demo")   # change if you want


def substitute_placeholders(compiled, variables):
    """Simple fallback template substitution."""
    if isinstance(compiled, str):
        for k, v in variables.items():
            compiled = compiled.replace("{{" + k + "}}", str(v)).replace("{" + k + "}", str(v))
    elif isinstance(compiled, list):
        for msg in compiled:
            if isinstance(msg.get("content"), str):
                for k, v in variables.items():
                    msg["content"] = msg["content"].replace("{{" + k + "}}", str(v)).replace("{" + k + "}", str(v))
    return compiled


# 3️⃣ --- Global model (so Weave op doesn't need to accept a model object) ---
model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.2)
lf_cb = CallbackHandler()


# 4️⃣ --- Weave-op for the actual LLM call ---
@weave.op()
def traced_model_call(
    compiled_prompt,
    config,
    prompt_name: str,
    label: str,
    variables: dict,
    prompt_version,
    model_name: str,
):
    """
    This function becomes a node in the Weave trace.
    Inside it, the underlying OpenAI call (via LangChain) will also be auto-traced by Weave.
    """
    start_time = time.time()
    resp = model.invoke(compiled_prompt, config=config)
    end_time = time.time()
    latency_ms = (end_time - start_time) * 1000.0

    # Try to get token usage (LangChain + OpenAI)
    token_usage = {}
    try:
        token_usage = resp.response_metadata.get("token_usage", {})
    except Exception:
        token_usage = {}

    # Log basic metrics to wandb for nice charts
    wandb.log(
        {
            "latency_ms": latency_ms,
            "prompt_name": prompt_name,
            "label": label,
            "prompt_version": prompt_version,
            "model_name": model_name,
            "prompt_tokens": token_usage.get("prompt_tokens"),
            "completion_tokens": token_usage.get("completion_tokens"),
            "total_tokens": token_usage.get("total_tokens"),
            # log some of the variables as dimensions
            **{f"var_{k}": v for k, v in variables.items()},
        }
    )

    # Return only things that are easy to serialize for Weave
    return {
        "output_text": resp.content,
        "latency_ms": latency_ms,
        "token_usage": token_usage,
    }


def run_prompt(
    prompt_name: str,
    label: str,
    variables: dict,
):
    lf_client = get_client()
    prompt = lf_client.get_prompt(prompt_name, label=label)

    compiled = prompt.compile(variables=variables)
    compiled = substitute_placeholders(compiled, variables)

    if DEBUG_PREVIEW:
        if isinstance(compiled, list):
            preview = "\n".join(
                f"{m.get('role','?')}: {str(m.get('content',''))[:240]}"
                for m in compiled
            )
        else:
            preview = str(compiled)[:600]
        print(f"\n--- PREVIEW: {prompt_name}:{label} ---\n{preview}\n--- END PREVIEW ---")

    trace_id = f"{prompt_name}-{label}-{uuid.uuid4()}"
    config = {
        "callbacks": [lf_cb],
        "langfuse": {
            "trace_args": {
                "id": trace_id,
                "name": f"{prompt_name}:{label}",
                "user_id": "udara-local",
                "tags": [f"prompt:{prompt_name}", f"label:{label}"],
                "metadata": {
                    "prompt_version": getattr(prompt, "version", None),
                },
            }
        },
    }

    # Call the Weave-op wrapper instead of model.invoke directly
    result = traced_model_call(
        compiled_prompt=compiled,
        config=config,
        prompt_name=prompt_name,
        label=label,
        variables=variables,
        prompt_version=getattr(prompt, "version", None),
        model_name=getattr(model, "model_name", None),
    )

    print(f"\n=== {prompt_name}:{label} ===")
    print(result["output_text"])

    lf_client.flush()
    return result


# 5️⃣ --- Example vars & run ---
triage_vars = {
    "issue_text": "I was charged twice for the same order yesterday. Please reverse one of the payments.",
    "customer_tier": "Gold",
    "recent_orders": "Order#94821 on 2025-11-02; Visa **** 2219",
}

if __name__ == "__main__":
    run_prompt("demo-ticket-triage", "production", triage_vars)


In [ ]:
import os
from typing import Annotated, TypedDict, Optional, Literal

import weave
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


# --------- 0. ENV + INIT ---------
# Make sure these are set in your environment:
#   export OPENAI_API_KEY="sk-..."
#   export WANDB_API_KEY="wandb_..."
# Weave uses the WANDB_API_KEY under the hood.

weave.init(
    project="langgraph-weave-agents-demo",
    settings={
        "print_call_link": True,  # prints a URL to each trace in terminal
    },
)

# One shared model for simplicity
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)


# --------- 1. STATE DEFINITION ---------
class AgentState(TypedDict):
    # Conversation messages
    messages: Annotated[list[AnyMessage], add_messages]
    # Routing decision: "billing", "order", "chitchat"
    route: Optional[str]


# --------- 2. TOOLS (decorated so they appear in traces) ---------
@weave.op()
def get_order_status(order_id: str) -> str:
    # Fake implementation – just something visible in traces
    return f"Order {order_id} was delivered yesterday and payment was confirmed."


@weave.op()
def get_refund_policy() -> str:
    # Fake implementation – static policy text
    return (
        "Refunds are available within 30 days of purchase if the item is unused and in original packaging."
    )


# --------- 3. NODES (AGENTS) ---------

@weave.op()
def classify_intent(state: AgentState) -> AgentState:
    """
    Reads the latest user message and decides:
      - 'billing'
      - 'order'
      - 'chitchat'
    """
    messages = state["messages"]
    last_user = messages[-1].content if messages else ""

    system_msg = (
        "You are an intent classifier for a support assistant. "
        "Look at the user message and respond with EXACTLY one word:\n"
        "- billing  (if about payments, refunds, charges, invoices, etc.)\n"
        "- order    (if about order status, shipping, tracking, etc.)\n"
        "- chitchat (anything else)\n"
    )

    response = llm.invoke(
        [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": last_user},
        ]
    )

    text = response.content.strip().lower()
    if "bill" in text:
        route = "billing"
    elif "order" in text or "shipping" in text or "delivery" in text:
        route = "order"
    elif "chitchat" in text:
        route = "chitchat"
    else:
        # fallback
        route = "chitchat"

    print(f"[classify_intent] route = {route}")

    return {
        "messages": messages,  # unchanged
        "route": route,
    }


@weave.op()
def billing_agent(state: AgentState) -> AgentState:
    """
    Handles billing / payment / refund questions.
    Uses the get_refund_policy() tool and then LLM for a nice answer.
    """
    messages = state["messages"]
    last_user = messages[-1].content if messages else ""

    policy = get_refund_policy()

    system_msg = (
        "You are a helpful billing support agent. Use the given refund policy "
        "and answer the user's question in a concise, friendly way."
    )

    response = llm.invoke(
        [
            {"role": "system", "content": system_msg},
            {
                "role": "system",
                "content": f"Refund policy: {policy}",
            },
            {"role": "user", "content": last_user},
        ]
    )

    ai_msg = AIMessage(content=response.content)
    print("[billing_agent] replying with billing answer.")
    return {
        "messages": [ai_msg],
        "route": state["route"],
    }


@weave.op()
def order_agent(state: AgentState) -> AgentState:
    """
    Handles order status questions.
    Extracts a fake order id (if any) and calls get_order_status().
    """
    messages = state["messages"]
    last_user = messages[-1].content if messages else ""

    # Super naive "extraction": just search for a number.
    import re

    match = re.search(r"\d{5,}", last_user)
    order_id = match.group(0) if match else "12345"

    status = get_order_status(order_id)

    system_msg = (
        "You are an order support agent. Use the given order status info to answer clearly and briefly."
    )

    response = llm.invoke(
        [
            {"role": "system", "content": system_msg},
            {"role": "system", "content": f"Order status: {status}"},
            {"role": "user", "content": last_user},
        ]
    )

    ai_msg = AIMessage(content=response.content)
    print(f"[order_agent] replying about order {order_id}.")
    return {
        "messages": [ai_msg],
        "route": state["route"],
    }


@weave.op()
def chitchat_agent(state: AgentState) -> AgentState:
    """
    Fallback small-talk agent.
    """
    messages = state["messages"]
    last_user = messages[-1].content if messages else ""

    system_msg = "You are a friendly assistant for casual conversation. Reply briefly and nicely."

    response = llm.invoke(
        [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": last_user},
        ]
    )

    ai_msg = AIMessage(content=response.content)
    print("[chitchat_agent] replying as chitchat.")
    return {
        "messages": [ai_msg],
        "route": state["route"],
    }


# --------- 4. BUILD THE LANGGRAPH ---------

graph = StateGraph(AgentState)

graph.add_node("classify_intent", classify_intent)
graph.add_node("billing_agent", billing_agent)
graph.add_node("order_agent", order_agent)
graph.add_node("chitchat_agent", chitchat_agent)

graph.add_edge(START, "classify_intent")


def route_decider(state: AgentState) -> Literal["billing_agent", "order_agent", "chitchat_agent"]:
    route = state.get("route", "chitchat")
    if route == "billing":
        return "billing_agent"
    elif route == "order":
        return "order_agent"
    else:
        return "chitchat_agent"


graph.add_conditional_edges(
    "classify_intent",
    route_decider,
    {
        "billing_agent": "billing_agent",
        "order_agent": "order_agent",
        "chitchat_agent": "chitchat_agent",
    },
)

graph.add_edge("billing_agent", END)
graph.add_edge("order_agent", END)
graph.add_edge("chitchat_agent", END)

app = graph.compile()


# --------- 5. RUN A FEW TESTS TO GET NICE TRACES ---------

if __name__ == "__main__":
    test_messages = [
        "I was charged twice for the same order. How do I get a refund?",
        "Where is my order 94821? It was supposed to arrive yesterday.",
        "Hey, how are you? What do you think about AI?",
    ]

    for i, user_text in enumerate(test_messages, start=1):
        print(f"\n\n===== TEST {i}: {user_text} =====")
        state: AgentState = {
            "messages": [HumanMessage(content=user_text)],
            "route": None,
        }
        final = app.invoke(state)

        print("=== FINAL ASSISTANT MESSAGE ===")
        for m in final["messages"]:
            print(f"{m.type.upper()}: {m.content}")
